# Fase 3 · M04a: Encoding AutoML (100% Numérico)

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M04a — Encoding AutoML |

---

## 🎯 Qué hace

Aplica encoding 100% numérico para AutoML: ordinal encoding en categóricas, sin strings. Genera el dataset listo para AutoGluon, PyCaret y H2O.

## 📋 Requisitos

- `data/03_features/df_expediente_features.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/03_features/df_exp_automl_target.parquet` | Dataset 100% numérico para AutoML |
| `docs/html/fase3/m04a_automl_target.html` | Informe HTML |

## 🔄 Flujo

```
df_expediente_features.parquet
    ↓ Ordinal encoding en categóricas
    ↓ Verificación sin strings
    → data/03_features/df_exp_automl_target.parquet + HTML
```

## ➡️ Siguiente

`f3_m04b_eda_target.ipynb` — encoding mixto para EDA


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# Detectar entorno
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np

from src.config import RUTA_FEATURES, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

# Rutas
RUTA_FASE3_HTML = RUTA_HTML / 'fase3'
crear_directorios([RUTA_FEATURES, RUTA_FASE3_HTML])

fmt = formato_numero_es
info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS
# ============================================================================

print('\n' + '='*70)
print('F3-M04: ENCODING AutoML (100% NUMÉRICO)')
print('='*70)

df = pd.read_parquet(RUTA_FEATURES / 'df_expediente_features.parquet')
n_registros_entrada = len(df)
n_cols_entrada = len(df.columns)

print(f'\n✓ Cargado: df_expediente_features.parquet')
print(f'  • Registros: {fmt(n_registros_entrada)}')
print(f'  • Columnas entrada: {n_cols_entrada}')


F3-M04: ENCODING AutoML (100% NUMÉRICO)

✓ Cargado: df_expediente_features.parquet
  • Registros: 33.621
  • Columnas entrada: 50


In [3]:
# ============================================================================
# CELDA 3: CODIFICAR STRINGS A NUMÉRICOS
# ============================================================================

print('\n' + '-'*70)
print('CODIFICANDO STRINGS A NÚMEROS')
print('-'*70)

VIA_ACCESO_MAP = {
    'Pruebas acceso Bachiller Logse': 10,
    'Ciclo Formativo de Grado sup. o equivalente': 5,
    'Extranjeros CEE': 11,
    'Titulados Universitarios': 4,
    'Pruebas acceso mayores 25 años': 7,
    'Pruebas acceso mayores 25años': 7,
    'Pruebas acceso mayores 40 años': 13,
    'Pruebas acceso mayores 40años': 13,
    'Pruebas acceso mayores 45 años': 12,
    'Pruebas acceso mayores 45años': 12,
    'Extranjeros no CEE': 6,
    'Sin datos': 0
}

UNI_ORIGEN_MAP = {'UJI': 40, 'UPV': 27, 'UV': 18, 'UA': 1, 'UMH': 55, np.nan: 0, None: 0}
RAMA_MAP = {'TE': 1, 'HU': 2, 'SO': 3, 'SA': 4, 'EX': 5}
SEXO_MAP = {'Hombre': 1, 'Mujer': 0}
PROVINCIA_MAP = {'Castelló': 1, 'Alacant': 2, 'Tarragona': 3, 'Terol': 4, 'València': 5, 'OTRAS': 0}
EGRESADO_MAP = {'S': 1, 'N': 0}
PAIS_REGION_MAP = {
    'España': 1, 'Portugal': 2, 'Francia': 2, 'Italia': 2, 'Alemania': 2,
    'Brasil': 3, 'México': 3, 'Colombia': 3, 'Argentina': 3,
    'China': 4, 'Japón': 4, 'Tailandia': 4,
    'Marruecos': 5, 'Argelia': 5, 'Túnez': 5
}

for col, mapping in [('rama', RAMA_MAP), ('sexo', SEXO_MAP), ('via_acceso', VIA_ACCESO_MAP),
                      ('provincia', PROVINCIA_MAP), ('pais_nombre', PAIS_REGION_MAP),
                      ('universidad_origen', UNI_ORIGEN_MAP), ('egresado', EGRESADO_MAP)]:
    if col in df.columns:
        df[col] = df[col].map(mapping).fillna(0).astype(int)
        print(f'✓ {col}')


# --- Fix: situacion_laboral (texto pendiente de codificar) ---
if 'situacion_laboral' in df.columns:
    from sklearn.preprocessing import LabelEncoder
    df['situacion_laboral'] = df['situacion_laboral'].fillna('desconocido')
    le = LabelEncoder()
    df['situacion_laboral'] = le.fit_transform(df['situacion_laboral'])
    print(f'✓ situacion_laboral (LabelEncoder: {len(le.classes_)} categorías)')

# Booleans → 0/1
bool_cols = df.select_dtypes(include='bool').columns.tolist()
for col in bool_cols:
    df[col] = df[col].astype(int)
print(f'\n✓ Booleans → 0/1 ({len(bool_cols)} cols)')

# Redondear floats
float_cols = df.select_dtypes(include=['float64', 'float32']).columns.tolist()
for col in float_cols:
    df[col] = df[col].round(2)
print(f'✓ Floats redondeados ({len(float_cols)} cols)')


----------------------------------------------------------------------
CODIFICANDO STRINGS A NÚMEROS
----------------------------------------------------------------------
✓ rama
✓ sexo
✓ via_acceso
✓ provincia
✓ pais_nombre
✓ universidad_origen
✓ egresado


✓ situacion_laboral (LabelEncoder: 12 categorías)

✓ Booleans → 0/1 (6 cols)
✓ Floats redondeados (19 cols)


In [4]:
# ============================================================================
# CELDA 4: ELIMINAR COLUMNAS
# ============================================================================

print('\n' + '-'*70)
print('ELIMINANDO COLUMNAS')
print('-'*70)

cols_eliminar = ['titulacion', 'fecha_nacimiento', 'poblacion', 'cupo']
cols_a_eliminar = [col for col in cols_eliminar if col in df.columns]

for col in cols_a_eliminar:
    df = df.drop(col, axis=1)
    print(f'✓ {col}')

print(f'\nTotal: {len(cols_a_eliminar)} eliminadas')


----------------------------------------------------------------------
ELIMINANDO COLUMNAS
----------------------------------------------------------------------
✓ titulacion
✓ fecha_nacimiento
✓ poblacion
✓ cupo

Total: 4 eliminadas


In [5]:
# ============================================================================
# CELDA 5: VERIFICACIÓN Y ESTADÍSTICAS
# ============================================================================

print('\n' + '-'*70)
print('VERIFICACIÓN FINAL')
print('-'*70)

n_registros_salida = len(df)
n_cols_salida = len(df.columns)

print(f'\n✓ Registros: {fmt(n_registros_salida)}')
print(f'✓ Columnas: {n_cols_salida}')

tipos = df.dtypes.value_counts()
print(f'\nTipos de datos:')
for dtype, count in tipos.items():
    print(f'  • {dtype}: {count}')

no_num = df.select_dtypes(exclude=['number']).columns.tolist()
if no_num:
    print(f'\n🚨 NO NUMÉRICOS: {no_num}')
else:
    print(f'\n✅ 100% NUMÉRICO')


----------------------------------------------------------------------
VERIFICACIÓN FINAL
----------------------------------------------------------------------

✓ Registros: 33.621
✓ Columnas: 46

Tipos de datos:
  • int64: 27
  • float64: 19

✅ 100% NUMÉRICO


In [6]:
# ============================================================================
# CELDA 6: GUARDAR DATASET
# ============================================================================

print('\n' + '='*70)
print('GUARDANDO')
print('='*70)

ruta_salida = RUTA_FEATURES / 'df_exp_automl_target.parquet'
df.to_parquet(ruta_salida, index=False)
tamanio_mb = ruta_salida.stat().st_size / 1024 / 1024

print(f'\n✓ {ruta_salida.name}')
print(f'  • Registros: {fmt(n_registros_salida)}')
print(f'  • Columnas: {n_cols_salida}')
print(f'  • Tamaño: {tamanio_mb:.1f} MB')


GUARDANDO

✓ df_exp_automl_target.parquet
  • Registros: 33.621
  • Columnas: 46
  • Tamaño: 1.1 MB


In [7]:
# ============================================================================
# CELDA 7: GENERAR HTML
# ============================================================================

print('\n' + '='*70)
print('GENERANDO HTML')
print('='*70)

nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa='fase3',
    modulo_activo='m04'
)

# KPIs
KPIS = [
    {'valor': fmt(n_registros_salida), 'titulo': 'Expedientes'},
    {'valor': str(n_cols_salida), 'titulo': 'Columnas'},
    {'valor': '100%', 'titulo': 'Numérico'},
    {'valor': '✓', 'titulo': 'AutoML Listo'},
]
kpis_html = generar_kpis_html(KPIS)

# S1: Transformación
s1 = generar_seccion_html('Transformación', f'''
<div style="display:grid;grid-template-columns:1fr auto 1fr;gap:20px;align-items:center;text-align:center;">
    <div style="background:#ebf8ff;padding:20px;border-radius:10px;">
        <div style="font-size:28px;font-weight:bold;color:#3182ce;">{n_cols_entrada}</div>
        <div style="color:#2c5282;">columnas entrada</div>
    </div>
    <div style="font-size:48px;color:#a0aec0;">→</div>
    <div style="background:#f0fff4;padding:20px;border-radius:10px;">
        <div style="font-size:28px;font-weight:bold;color:#38a169;">{n_cols_salida}</div>
        <div style="color:#276749;">100% numérico</div>
    </div>
</div>
<p style="text-align:center;margin-top:15px;">Eliminadas: {', '.join(cols_a_eliminar)} | Codificadas: 7 variables</p>
''', '⚙️')

# S2: Codificaciones
codif_html = ''
for name, mapping in [('rama', RAMA_MAP), ('sexo', SEXO_MAP), ('egresado', EGRESADO_MAP)]:
    items = ' | '.join([f'{k}→{v}' for k,v in list(mapping.items())[:3]])
    codif_html += f'<div style="margin-bottom:10px;"><strong>{name}:</strong> {items}...</div>'

s2 = generar_seccion_html('Codificaciones Aplicadas', f'''
{codif_html}
<p style="margin-top:15px;padding:10px;background:#f7fafc;border-radius:5px;color:#4a5568;">
    Diccionarios: via_acceso (0-13) | provincia (0-5) | pais (1-5 regiones) | universidad_origen (códigos oficiales)
</p>
''', '🔤')

# S3: Tipos de datos
s3 = generar_seccion_html('Estructura Final', f'''
<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:15px;">
    <div style="background:#ebf8ff;padding:15px;border-radius:8px;text-align:center;">
        <div style="font-size:24px;font-weight:bold;color:#3182ce;">{sum(1 for t in df.dtypes if 'int' in str(t))}</div>
        <div style="font-size:12px;color:#2c5282;">int64 (enteros)</div>
    </div>
    <div style="background:#f0fff4;padding:15px;border-radius:8px;text-align:center;">
        <div style="font-size:24px;font-weight:bold;color:#38a169;">{sum(1 for t in df.dtypes if 'float' in str(t))}</div>
        <div style="font-size:12px;color:#276749;">float64 (reales 2 dec)</div>
    </div>
    <div style="background:#fffaf0;padding:15px;border-radius:8px;text-align:center;">
        <div style="font-size:24px;font-weight:bold;color:#ed8936;">0</div>
        <div style="font-size:12px;color:#c05621;">strings/object</div>
    </div>
</div>
''', '📊')

contenido_html = kpis_html + s1 + s2 + s3

html_completo = render_base_html(
    titulo='M04: Encoding AutoML',
    icono='⚙️',
    subtitulo='Fase 3: Feature Engineering | TFM Abandono Universitario',
    nav_fases=nav_fases_html,
    nav_modulos=nav_modulos_html,
    contenido=contenido_html,
    notebook_nombre='f3_m04_automl_target.ipynb',
    notebook_carpeta='fase3_features'
)

ruta_html = RUTA_FASE3_HTML / 'm04_automl.html'
guardar_html(html_completo, ruta_html)
print(f'🌐 HTML: {ruta_html}')


GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m04_automl.html
🌐 HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m04_automl.html


In [8]:
# ============================================================================
# CELDA 8: RESUMEN FINAL
# ============================================================================

print('\n' + '='*70)
print('✅ F3-M04 COMPLETADO')
print('='*70)
print(f'📥 Entrada: {n_cols_entrada} columnas (mixto)')
print(f'📤 Salida: {n_cols_salida} columnas (100% numérico)')
print(f'💾 {ruta_salida}')
print(f'🌐 {ruta_html}')
print(f'\n📌 Siguiente: f3_m04b_eda_target.ipynb')


✅ F3-M04 COMPLETADO
📥 Entrada: 50 columnas (mixto)
📤 Salida: 46 columnas (100% numérico)
💾 C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\df_exp_automl_target.parquet
🌐 C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m04_automl.html

📌 Siguiente: f3_m04b_eda_target.ipynb
